# M3 Data Profiling & Quality Investigation
This notebook profiles the M2 database (`data/aviation.db`) across all six data quality families before building the cleaned layer.


In [2]:
import sqlite3
import pandas as pd

con = sqlite3.connect('../data/aviation.db')

tables = ['aircraft', 'flight_state', 'location', 'weather_reading', 'quarantine']
counts = {t: pd.read_sql_query(f"SELECT COUNT(*) AS cnt FROM {t}", con)['cnt'].iloc[0] for t in tables}
df_counts = pd.DataFrame(list(counts.items()), columns=['Table', 'Row_Count'])
print(df_counts.to_string(index=False))


          Table  Row_Count
       aircraft       8104
   flight_state       8026
       location          1
weather_reading        168
     quarantine         78


## Family 1: Missing Values & Mechanisms
Checking missingness across `aircraft`, `flight_state`, and `weather_reading`, and testing whether `baro_altitude` missingness depends on `on_ground` (MAR mechanism).


In [4]:
df_fs = pd.read_sql_query("SELECT * FROM flight_state", con)
df_ac = pd.read_sql_query("SELECT * FROM aircraft", con)
df_wr = pd.read_sql_query("SELECT * FROM weather_reading", con)

print("--- FLIGHT_STATE MISSINGNESS ---")
missing_fs = df_fs.isna().sum().to_frame('Missing_Count')
missing_fs['Missing_Share_%'] = (missing_fs['Missing_Count'] / len(df_fs) * 100).round(2)
print(missing_fs)

print("\n--- ALTITUDE MISSINGNESS BY ON_GROUND (MAR EVIDENCE) ---")
mar_check = df_fs.groupby('on_ground')['baro_altitude'].apply(lambda s: pd.Series({
    'Total_Rows': len(s),
    'Missing_Altitude': s.isna().sum(),
    'Missing_Share_%': round(s.isna().mean() * 100, 2)
})).unstack()
print(mar_check)

print("\n--- AIRCRAFT EMPTY/NULL CALLSIGNS ---")
empty_cs = (df_ac['callsign'].isna() | (df_ac['callsign'] == '')).sum()
print(f"Empty or NULL callsigns: {empty_cs} / {len(df_ac)} ({round(empty_cs/len(df_ac)*100, 2)}%)")


--- FLIGHT_STATE MISSINGNESS ---
               Missing_Count  Missing_Share_%
aircraft_key               0             0.00
observed_at                0             0.00
longitude                  0             0.00
latitude                   0             0.00
baro_altitude            823            10.25
velocity                   1             0.01
on_ground                  0             0.00
retrieved_at               0             0.00

--- ALTITUDE MISSINGNESS BY ON_GROUND (MAR EVIDENCE) ---
           Total_Rows  Missing_Altitude  Missing_Share_%
on_ground                                               
0              7177.0               3.0             0.04
1               849.0             820.0            96.58

--- AIRCRAFT EMPTY/NULL CALLSIGNS ---
Empty or NULL callsigns: 184 / 8104 (2.27%)


## Family 2 & 3: Invalid Values (Sentinels/Rules) and Unusual Values (Robust Z-Score / IQR)
Checking min, median, max, impossible values (supersonic speeds, negative altitudes, sentinel callsigns), and robust outliers using Median and MAD.


In [6]:
print("--- SUMMARY STATS (FLIGHT_STATE) ---")
for col in ['longitude', 'latitude', 'baro_altitude', 'velocity']:
    s = df_fs[col].dropna()
    med = s.median()
    mad = (s - med).abs().median()
    q25, q75 = s.quantile(0.25), s.quantile(0.75)
    iqr = q75 - q25
    print(f"{col:15s} | min: {s.min():8.2f} | median: {med:8.2f} | max: {s.max():8.2f} | MAD: {mad:6.2f} | IQR: {iqr:7.2f}")

print("\n--- INVALID VELOCITY (> 350 m/s for commercial subsonic) ---")
print(df_fs[df_fs['velocity'] > 350][['aircraft_key', 'observed_at', 'velocity', 'baro_altitude', 'on_ground']])

print("\n--- NEGATIVE BARO_ALTITUDE (< 0 m) ---")
print(f"Count of negative altitudes: {(df_fs['baro_altitude'] < 0).sum()}")
print(df_fs[df_fs['baro_altitude'] < 0][['aircraft_key', 'observed_at', 'baro_altitude', 'on_ground']].head(10))

print("\n--- SENTINEL / PLACEHOLDER CALLSIGNS ---")
sentinels = df_ac[df_ac['callsign'].isin(['00000000', '0000000', 'UNIDENT', 'TEST', '999999'])]
print(sentinels[['aircraft_key', 'icao24', 'callsign', 'origin_country']])

print("\n--- WEATHER_READING STATS & CHECKS ---")
print(df_wr[['temperature_2m', 'wind_speed_10m', 'precipitation']].describe().round(2))
print("Negative wind speed:", (df_wr['wind_speed_10m'] < 0).sum())
print("Negative precipitation:", (df_wr['precipitation'] < 0).sum())


--- SUMMARY STATS (FLIGHT_STATE) ---
longitude       | min:  -172.90 | median:     7.91 | max:   179.69 | MAD:  18.69 | IQR:   36.86
latitude        | min:   -45.12 | median:    41.29 | max:    69.98 | MAD:   7.05 | IQR:   14.10
baro_altitude   | min:  -152.40 | median:  9144.00 | max: 20604.48 | MAD: 2438.40 | IQR: 7924.80
velocity        | min:     0.00 | median:   208.18 | max:  2160.22 | MAD:  38.28 | IQR:  141.92

--- INVALID VELOCITY (> 350 m/s for commercial subsonic) ---
      aircraft_key           observed_at  velocity  baro_altitude  on_ground
3185          3218  2026-10-01T08:55:11Z   2160.22        9060.18          0
3848          3887  2026-10-01T10:23:49Z    914.97       10660.38          0
7039          7109  2026-10-01T09:51:12Z   1390.56       11582.40          0

--- NEGATIVE BARO_ALTITUDE (< 0 m) ---
Count of negative altitudes: 45
      aircraft_key           observed_at  baro_altitude  on_ground
152            153  2026-10-01T10:32:41Z         -68.58          0
31

## Family 4: Failures Across Rows or Fields
Checking cross-field contradictions (e.g. `on_ground == 1` with high altitude or high velocity) and sequence checks on hourly weather readings.


In [8]:
ground_high_alt = df_fs[(df_fs['on_ground'] == 1) & (df_fs['baro_altitude'] > 1000)]
ground_high_vel = df_fs[(df_fs['on_ground'] == 1) & (df_fs['velocity'] > 100)]
air_zero_vel = df_fs[(df_fs['on_ground'] == 0) & (df_fs['velocity'] == 0)]

print(f"On ground but baro_altitude > 1000m: {len(ground_high_alt)}")
print(f"On ground but velocity > 100 m/s:    {len(ground_high_vel)}")
print(f"In air (on_ground=0) but velocity=0: {len(air_zero_vel)}")
if len(air_zero_vel) > 0:
    print(air_zero_vel[['aircraft_key', 'observed_at', 'baro_altitude', 'velocity', 'on_ground']].head(10))

# Weather sequence stuck check (identical consecutive values)
df_wr_sorted = df_wr.sort_values('observed_at')
for col in ['temperature_2m', 'wind_speed_10m']:
    runs = df_wr_sorted[col].groupby((df_wr_sorted[col] != df_wr_sorted[col].shift()).cumsum()).size()
    print(f"Max consecutive identical hourly values for {col}: {runs.max()} hours")


On ground but baro_altitude > 1000m: 1
On ground but velocity > 100 m/s:    0
In air (on_ground=0) but velocity=0: 4
      aircraft_key           observed_at  baro_altitude  velocity  on_ground
483            487  2026-10-01T10:30:07Z         121.92       0.0          0
2390          2413  2026-10-01T10:32:53Z         243.84       0.0          0
2516          2542  2026-10-01T10:30:48Z         -38.10       0.0          0
2625          2652  2026-10-01T10:32:06Z          30.48       0.0          0
Max consecutive identical hourly values for temperature_2m: 4 hours
Max consecutive identical hourly values for wind_speed_10m: 3 hours


## Family 5 & 6: Text/Category Variants (`repr()`) and Identity Collisions
Inspecting distinct `origin_country` strings with `repr()` to catch encoding issues, and checking duplicate active `callsign` values across different `icao24` transponders.


In [10]:
print("--- DISTINCT ORIGIN COUNTRIES WITH REPR() (NON-ASCII OR LONG NAMES) ---")
for c in sorted(df_ac['origin_country'].dropna().unique()):
    if not c.isascii() or len(c) > 20 or " " in c:
        print(f"{repr(c):45s} -> count: {(df_ac['origin_country'] == c).sum()}")

print("\n--- IDENTITY CHECK: DUPLICATE CALLSIGNS ACROSS DIFFERENT ICAO24 ---")
valid_cs = df_ac[df_ac['callsign'].notna() & (df_ac['callsign'] != '')]
dup_cs = valid_cs[valid_cs.duplicated(subset=['callsign'], keep=False)].sort_values('callsign')
print(f"Total rows sharing a callsign: {len(dup_cs)}")
print(dup_cs[['aircraft_key', 'icao24', 'callsign', 'origin_country']].head(15).to_string(index=False))


--- DISTINCT ORIGIN COUNTRIES WITH REPR() (NON-ASCII OR LONG NAMES) ---
'Czech Republic'                              -> count: 67
"Côte d'Ivoire"                               -> count: 1
'Dominican Republic'                          -> count: 3
'Islamic Republic of Iran'                    -> count: 3
'Kingdom of the Netherlands'                  -> count: 126
'Libyan Arab Jamahiriya'                      -> count: 1
'New Zealand'                                 -> count: 17
'Republic of Korea'                           -> count: 135
'Republic of Moldova'                         -> count: 5
'Russian Federation'                          -> count: 35
'San Marino'                                  -> count: 19
'Saudi Arabia'                                -> count: 42
'South Africa'                                -> count: 43
'Sri Lanka'                                   -> count: 5
'Syrian Arab Republic'                        -> count: 1
'The former Yugoslav Republic of Macedonia'   ->